# Airbnb Madrid: data cleaning, analysis and price recommendation

Data: Inside Airbnb listings for Madrid (`data/listings.csv.gz`).

## 1. Setup

In [ ]:
import sqlite3

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

print("All set | pandas", pd.__version__)

## 2. Load the data

In [ ]:
df = pd.read_csv("data/listings.csv.gz")
print(df.shape)
df.head()

In [ ]:
df[["id", "neighbourhood_cleansed", "room_type", "accommodates", "bedrooms", "price", "number_of_reviews", "review_scores_rating"]].head()

## 3. Data quality check

In [ ]:
print(df["price"].head())
print("Data type:", df["price"].dtype)
print()
print("Missing values (%):")
cols = ["price", "bedrooms", "review_scores_rating", "neighbourhood_cleansed", "room_type", "accommodates"]
print((df[cols].isna().mean() * 100).round(1))

## 4. Cleaning

In [ ]:
# 1. Convert price to a number
df["price"] = (
    df["price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .astype(float)
)

# 2. Keep only rows that have a price
rows_before = len(df)
df_clean = df.dropna(subset=["price"]).copy()
print(f"Rows before: {rows_before} | after: {len(df_clean)}")

# 3. Price summary
print(df_clean["price"].describe(percentiles=[.01, .25, .5, .75, .99]))

# 4. Why are ratings missing?
no_rating = df_clean[df_clean["review_scores_rating"].isna()]
print()
print("Listings without a rating:", len(no_rating))
print("Of those, with 0 reviews (%):", round((no_rating["number_of_reviews"] == 0).mean() * 100, 1))

In [ ]:
# 1. Remove extreme prices
before = len(df_clean)
df_clean = df_clean[(df_clean["price"] >= 20) & (df_clean["price"] <= 1000)].copy()
print(f"Removed as extreme prices: {before - len(df_clean)} | remaining: {len(df_clean)}")

# 2. Feature: does the listing have reviews?
df_clean["has_reviews"] = (df_clean["number_of_reviews"] > 0).astype(int)

# 3. Feature: log of price
df_clean["log_price"] = np.log(df_clean["price"])

# 4. Compare the distribution of price and log price
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
df_clean["price"].hist(bins=50, ax=ax[0])
ax[0].set_title("Price")
df_clean["log_price"].hist(bins=50, ax=ax[1])
ax[1].set_title("Log price")
plt.show()

## 5. SQL analysis

In [ ]:
# 1. Select the columns we will use
cols = [
    "id", "neighbourhood_cleansed", "room_type", "property_type",
    "accommodates", "bedrooms", "beds", "price", "log_price",
    "number_of_reviews", "review_scores_rating", "has_reviews",
    "availability_365", "host_is_superhost", "latitude", "longitude"
]
df_sql = df_clean[cols]

# 2. Create the database and save the table
conn = sqlite3.connect("airbnb.db")
df_sql.to_sql("listings", conn, if_exists="replace", index=False)

# 3. Check it was saved
print(pd.read_sql("SELECT COUNT(*) AS total FROM listings", conn))

In [ ]:
# Top 10 most expensive neighbourhoods (min. 50 listings)
query = """
SELECT
    neighbourhood_cleansed AS neighbourhood,
    COUNT(*) AS num_listings,
    ROUND(AVG(price), 1) AS avg_price
FROM listings
GROUP BY neighbourhood_cleansed
HAVING COUNT(*) >= 50
ORDER BY avg_price DESC
LIMIT 10
"""
pd.read_sql(query, conn)

In [ ]:
# Price and capacity by room type
query = """
SELECT
    room_type,
    COUNT(*) AS num_listings,
    ROUND(AVG(price), 1) AS avg_price,
    ROUND(AVG(accommodates), 1) AS avg_capacity
FROM listings
GROUP BY room_type
ORDER BY num_listings DESC
"""
pd.read_sql(query, conn)

In [ ]:
# Superhosts vs. regular hosts
query = """
SELECT
    host_is_superhost AS superhost,
    COUNT(*) AS num_listings,
    ROUND(AVG(price), 1) AS avg_price,
    ROUND(AVG(review_scores_rating), 2) AS avg_rating
FROM listings
GROUP BY host_is_superhost
"""
result = pd.read_sql(query, conn)
conn.close()
result

## 6. Regression models (statsmodels)

In [ ]:
# 1. Prepare the model data
df_model = df_sql.dropna(subset=["bedrooms", "host_is_superhost"]).copy()
df_model["superhost"] = (df_model["host_is_superhost"] == "t").astype(int)
df_model = df_model[df_model["room_type"] != "Hotel room"]
print("Observations:", len(df_model))

# 2. Fit the model (heteroskedasticity-robust standard errors)
model1 = smf.ols(
    "log_price ~ accommodates + bedrooms + C(room_type) + superhost + has_reviews",
    data=df_model
).fit(cov_type="HC3")

# 3. Show key results
print("R2:", round(model1.rsquared, 3))
print(model1.summary().tables[1])

In [ ]:
# Same model + neighbourhood fixed effects
model2 = smf.ols(
    "log_price ~ accommodates + bedrooms + C(room_type) + superhost + has_reviews + C(neighbourhood_cleansed)",
    data=df_model
).fit(cov_type="HC3")

print("R2 model 1:", round(model1.rsquared, 3), "| adjusted:", round(model1.rsquared_adj, 3))
print("R2 model 2:", round(model2.rsquared, 3), "| adjusted:", round(model2.rsquared_adj, 3))

key_vars = ["accommodates", "bedrooms", "superhost", "has_reviews",
            "C(room_type)[T.Private room]", "C(room_type)[T.Shared room]"]
comparison = pd.DataFrame({
    "model1": model1.params[key_vars],
    "model2": model2.params[key_vars]
}).round(4)
comparison

## 7. Machine learning: linear regression vs. Random Forest

In [ ]:
# 1. Features (dummy variables for the categories)
X = pd.get_dummies(
    df_model[["accommodates", "bedrooms", "superhost", "has_reviews",
              "availability_365", "latitude", "longitude",
              "room_type", "neighbourhood_cleansed"]],
    columns=["room_type", "neighbourhood_cleansed"], drop_first=True
).astype(float)
y = df_model["log_price"]

# 2. Train / test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Train both models (the Random Forest may take a minute)
lin = LinearRegression().fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=3,
                           n_jobs=-1, random_state=42).fit(X_train, y_train)

# 4. Evaluate on unseen data
for name, m in [("Linear regression", lin), ("Random Forest", rf)]:
    pred = m.predict(X_test)
    rmse_log = np.sqrt(mean_squared_error(y_test, pred))
    mae_eur = mean_absolute_error(np.exp(y_test), np.exp(pred))
    print(f"{name}: test R2 = {m.score(X_test, y_test):.3f} | "
          f"RMSE (log) = {rmse_log:.3f} | mean absolute error = {mae_eur:.1f} €")

# 5. Most important features according to the Random Forest
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print()
print(imp.head(10).round(3))

In [ ]:
# 1. Overfitting? Compare train vs. test R2
print(f"Random Forest | train R2 = {rf.score(X_train, y_train):.3f} | test R2 = {rf.score(X_test, y_test):.3f}")

# 2. Random Forest without availability_365
cols_no_avail = [c for c in X.columns if c != "availability_365"]
rf_no_avail = RandomForestRegressor(n_estimators=200, min_samples_leaf=3,
                                    n_jobs=-1, random_state=42).fit(X_train[cols_no_avail], y_train)
pred = rf_no_avail.predict(X_test[cols_no_avail])
rmse_log = np.sqrt(mean_squared_error(y_test, pred))
mae_eur = mean_absolute_error(np.exp(y_test), np.exp(pred))
print(f"RF without availability | test R2 = {rf_no_avail.score(X_test[cols_no_avail], y_test):.3f} | "
      f"RMSE (log) = {rmse_log:.3f} | mean absolute error = {mae_eur:.1f} €")

In [ ]:
# Hyperparameter tuning with cross-validation
configs = {
    "leaf=3": dict(min_samples_leaf=3),
    "leaf=10": dict(min_samples_leaf=10),
    "leaf=20": dict(min_samples_leaf=20),
    "leaf=10, max_features=0.5": dict(min_samples_leaf=10, max_features=0.5),
}

results = {}
for name, params in configs.items():
    m = RandomForestRegressor(n_estimators=100, n_jobs=-1, random_state=42, **params)
    cv = cross_val_score(m, X_train[cols_no_avail], y_train, cv=3, scoring="r2")
    results[name] = cv.mean()
    print(f"{name}: cross-validated R2 = {cv.mean():.3f}")

best = max(results, key=results.get)
print("\nBest configuration:", best)

## 8. Price recommendation tool

In [ ]:
# Final model: Random Forest, leaf=3, 200 trees, without availability_365
rf_final = rf_no_avail

# Uncertainty range from the test errors (in logs)
res = y_test - rf_final.predict(X_test[cols_no_avail])
q10, q90 = res.quantile([0.10, 0.90])

# Typical coordinates of each neighbourhood
coords = df_model.groupby("neighbourhood_cleansed")[["latitude", "longitude"]].median()

def recommend_price(neighbourhood, room_type, accommodates, bedrooms, superhost=0, has_reviews=0):
    row = pd.Series(0.0, index=cols_no_avail)
    row["accommodates"] = accommodates
    row["bedrooms"] = bedrooms
    row["superhost"] = superhost
    row["has_reviews"] = has_reviews
    row["latitude"], row["longitude"] = coords.loc[neighbourhood]
    for col in (f"room_type_{room_type}", f"neighbourhood_cleansed_{neighbourhood}"):
        if col in row.index:
            row[col] = 1.0
    pred = rf_final.predict(row.to_frame().T)[0]
    return round(np.exp(pred + q10)), round(np.exp(pred)), round(np.exp(pred + q90))

print("Sol, entire home, 4 guests, 2 bedrooms:", recommend_price("Sol", "Entire home/apt", 4, 2))
print("Universidad, private room, 2 guests, 1 bedroom:", recommend_price("Universidad", "Private room", 2, 1))

In [ ]:
# 1. What are the actual prices for these cases?
real_room = df_model[(df_model["neighbourhood_cleansed"] == "Universidad") &
                     (df_model["room_type"] == "Private room") &
                     (df_model["accommodates"] == 2)]
real_flat = df_model[(df_model["neighbourhood_cleansed"] == "Sol") &
                     (df_model["room_type"] == "Entire home/apt") &
                     (df_model["accommodates"] == 4)]
print("Private room in Universidad (2 guests): n =", len(real_room), "| actual median =", real_room["price"].median())
print("Entire home in Sol (4 guests): n =", len(real_flat), "| actual median =", real_flat["price"].median())

# 2. Does the recommendation change if the listing has reviews?
print()
print("Room, no reviews:  ", recommend_price("Universidad", "Private room", 2, 1, has_reviews=0))
print("Room, with reviews:", recommend_price("Universidad", "Private room", 2, 1, has_reviews=1))
print("Flat, no reviews:  ", recommend_price("Sol", "Entire home/apt", 4, 2, has_reviews=0))
print("Flat, with reviews:", recommend_price("Sol", "Entire home/apt", 4, 2, has_reviews=1))

In [ ]:
# 0. Check the hypothesis: how many listings without reviews are in those groups
for neighbourhood, rtype, cap in [("Universidad", "Private room", 2), ("Sol", "Entire home/apt", 4)]:
    sub = df_model[(df_model["neighbourhood_cleansed"] == neighbourhood) &
                   (df_model["room_type"] == rtype) & (df_model["accommodates"] == cap)]
    print(neighbourhood, rtype, cap, "->", sub.groupby("has_reviews").size().to_dict())

# 1. Model using only variables known before the listing is published
cols_new = [c for c in cols_no_avail if c not in ("has_reviews", "superhost")]
rf_new = RandomForestRegressor(n_estimators=200, min_samples_leaf=3,
                               n_jobs=-1, random_state=42).fit(X_train[cols_new], y_train)

pred = rf_new.predict(X_test[cols_new])
print()
print(f"RF new listing | test R2 = {rf_new.score(X_test[cols_new], y_test):.3f} | "
      f"mean absolute error = {mean_absolute_error(np.exp(y_test), np.exp(pred)):.1f} €")

# 2. Uncertainty range from this model's errors
q10, q90 = (y_test - pred).quantile([0.10, 0.90])

# 3. New function (no reviews or superhost inputs)
def recommend_price(neighbourhood, room_type, accommodates, bedrooms):
    row = pd.Series(0.0, index=cols_new)
    row["accommodates"] = accommodates
    row["bedrooms"] = bedrooms
    row["latitude"], row["longitude"] = coords.loc[neighbourhood]
    for col in (f"room_type_{room_type}", f"neighbourhood_cleansed_{neighbourhood}"):
        if col in row.index:
            row[col] = 1.0
    p = rf_new.predict(row.to_frame().T)[0]
    return round(np.exp(p + q10)), round(np.exp(p)), round(np.exp(p + q90))

print()
print("Sol, entire home, 4 guests, 2 bedrooms:", recommend_price("Sol", "Entire home/apt", 4, 2),
      "| actual median:", real_flat["price"].median())
print("Universidad, private room, 2 guests, 1 bedroom:", recommend_price("Universidad", "Private room", 2, 1),
      "| actual median:", real_room["price"].median())

## 9. Model validation

In [ ]:
groups = [("Sol", "Entire home/apt", 4), ("Universidad", "Private room", 2)]

for neighbourhood, rtype, cap in groups:
    group = df_model[(df_model["neighbourhood_cleansed"] == neighbourhood) &
                     (df_model["room_type"] == rtype) &
                     (df_model["accommodates"] == cap)]

    # 1. Actual price by number of bedrooms
    print(f"--- {neighbourhood}, {rtype}, {cap} guests ---")
    print(group.groupby("bedrooms")["price"].agg(["count", "median"]).round(1))

    # 2. Calibration: test listings only (unseen by the model)
    idx_test = group.index.intersection(X_test.index)
    pred_g = np.exp(rf_new.predict(X_test.loc[idx_test, cols_new]))
    print(f"In test (n={len(idx_test)}): predicted median = {np.median(pred_g):.1f} | "
          f"actual median = {group.loc[idx_test, 'price'].median():.1f}")
    print()

# 3. Sensitivity of the recommendation to the number of bedrooms
for b in [1, 2, 3]:
    print(f"Sol, entire home, 4 guests, {b} bedrooms:", recommend_price("Sol", "Entire home/apt", 4, b))

In [ ]:
pred_test = pd.Series(np.exp(rf_new.predict(X_test[cols_new])), index=X_test.index)
cal = pd.DataFrame({"pred": pred_test, "actual": np.exp(y_test)})
cal["decile"] = pd.qcut(cal["pred"], 10, labels=False) + 1
cal["in_range"] = (cal["actual"] >= cal["pred"] * np.exp(q10)) & (cal["actual"] <= cal["pred"] * np.exp(q90))

# 1. Calibration by decile of predicted price
print(cal.groupby("decile").agg(
    pred_median=("pred", "median"),
    actual_median=("actual", "median"),
    range_coverage=("in_range", "mean")
).round(2))

# 2. Calibration by room type and bedrooms
cal["room_type"] = df_model.loc[cal.index, "room_type"]
cal["bedrooms"] = df_model.loc[cal.index, "bedrooms"].clip(upper=4)
print()
print(cal.groupby(["room_type", "bedrooms"]).agg(
    n=("actual", "size"),
    pred_median=("pred", "median"),
    actual_median=("actual", "median")
).round(1))

## 10. Export for Tableau

In [ ]:
# 1. Combinations that exist in the data (min. 5 listings)
combos = (df_model.groupby(["neighbourhood_cleansed", "room_type", "accommodates", "bedrooms"])
          .size().reset_index(name="num_listings"))
combos = combos[(combos["num_listings"] >= 5) & (combos["bedrooms"] <= 5)].reset_index(drop=True)

# 2. Feature matrix in the same format as the model
M = pd.DataFrame(0.0, index=combos.index, columns=cols_new)
M["accommodates"] = combos["accommodates"]
M["bedrooms"] = combos["bedrooms"]
M["latitude"] = combos["neighbourhood_cleansed"].map(coords["latitude"])
M["longitude"] = combos["neighbourhood_cleansed"].map(coords["longitude"])
for col in cols_new:
    if col.startswith("room_type_"):
        M[col] = (combos["room_type"] == col.replace("room_type_", "")).astype(float)
    elif col.startswith("neighbourhood_cleansed_"):
        M[col] = (combos["neighbourhood_cleansed"] == col.replace("neighbourhood_cleansed_", "")).astype(float)

# 3. Predictions and range
p = rf_new.predict(M)
combos["min_price"] = np.exp(p + q10).round()
combos["recommended_price"] = np.exp(p).round()
combos["max_price"] = np.exp(p + q90).round()

listings_clean = df_model[["id", "neighbourhood_cleansed", "room_type", "accommodates", "bedrooms",
                           "price", "latitude", "longitude", "superhost", "has_reviews"]]

# 4. Export CSVs (";" separator, "," decimal and utf-8-sig so Excel and Tableau read them correctly)
csv_opts = dict(index=False, encoding="utf-8-sig", sep=";", decimal=",")
combos.to_csv("data/price_simulator.csv", **csv_opts)
listings_clean.to_csv("data/listings_clean.csv", **csv_opts)

# 5. Excel file with both tables (data source for the Tableau dashboard)
with pd.ExcelWriter("data/airbnb_madrid.xlsx") as w:
    listings_clean.to_excel(w, sheet_name="listings", index=False)
    combos.to_excel(w, sheet_name="simulator", index=False)

print(len(combos), "combinations exported")
combos.head()

In [ ]:
# Check: re-read the exported CSVs and make sure the price looks right
d = pd.read_csv("data/listings_clean.csv", sep=";", decimal=",", encoding="utf-8-sig")
s = pd.read_csv("data/price_simulator.csv", sep=";", decimal=",", encoding="utf-8-sig")
print(d["price"].describe().round(1))
print("Simulator rows:", len(s))